# Applications of DigiMicPy to Specific Systems 


Test: incorporating spatial elements (advanced usage of DigiMicPy)

In [1]:
# import packages 

import numpy as np
from scipy.stats import multivariate_normal
from scipy.integrate import solve_ivp
import os 

In [ ]:
# temperature-dependent uptake and respiration rates: parameters and functions 

def randtemp_param(N, kw): # generate random temperature-dependent traits for consumer species 
    rng=kw.get('rng',np.random) 

    L = kw['L'] # leakage 
    rho_t = kw['rho_t'] # correlation coefficient for covariance between activation energy and baseline uptake / mortality rates 
    L_v = np.mean(L)

    B0_m = kw.get("B0_m", -1.4954) # baseline mortality / respiration rate 
    B0_CUE = kw.get("B0_CUE", 0.1953) # baseline carbon use efficiency parameter
 
    B0_u = np.log(np.exp(B0_m) / (1 - L_v - B0_CUE)) # baseline uptake rate 
    B0 = np.array([B0_u, B0_m]) 

    B0_var_scale = kw.get("B0_var_scale", 0.17)
    B0_var = B0_var_scale * np.abs(B0)
    
    E_mean = kw.get(
    "E_mean",
    np.array([0.8146, 0.5741])
    ) # mean activation energy for uptake and respiration

    E_var_scale = kw.get("E_var_scale", 0.1364)
    E_var = E_var_scale * E_mean

    cov_xy = rho_t * np.sqrt(B0_var * E_var) # covariance between activation energy and baseline uptake / mortality rates

    cov_u = np.array([[B0_var[0], cov_xy[0]], [cov_xy[0], E_var[0]]]) # covariance matrix for uptake
    cov_m = np.array([[B0_var[1], cov_xy[1]], [cov_xy[1], E_var[1]]]) # covariance matrix for respiration

    allu = multivariate_normal.rvs(mean=[B0[0], E_mean[0]], cov=cov_u, size=N).T # draw random samples from multivariate normal distribution for uptake
    allm = multivariate_normal.rvs(mean=[B0[1], E_mean[1]], cov=cov_m, size=N).T # draw random samples from multivariate normal distribution for respiration

    B = np.column_stack((np.exp(allu[0]), np.exp(allm[0]))) # exponentiate the base rates to get the actual values
    E = np.column_stack((allu[1], allm[1])) # activation energy 

    Tp_mean = kw.get("Tp_mean", 273.15 + 35)
    Tp_sd   = kw.get("Tp_sd", 5)
    Tpu = rng.normal(Tp_mean, Tp_sd, N) # draw random peak temperatures for uptake from a normal distribution with mean and std set

    Tpm = Tpu + 3 # peak temperature for respiration is 3 degrees higher than for uptake
    Tp = np.column_stack((Tpu, Tpm)) 

    # temperature-dependent scaling of resource diffusion rates between patches

    D_ref = kw.get("D_ref", 0.05) # reference diffusion rate
    Tr = kw["Tr"]          # reference temperature
    T = kw.get("T", Tr)    # defaults to reference temperature
    D_resource = D_ref * (T / Tr) # diffusion rate calculation 

    return B, E, Tp, D_resource


def temp_trait(B, E, Tp, T, Tr, Ed):
    
    k = 0.0000862 

    # Arrhenius function with high-temp deactivation

    # uptake rate u(T)
    temp_u = B[:, 0] * np.exp((-E[:, 0] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 0] / (Ed - E[:, 0])) * np.exp(Ed / k * (1 / Tp[:, 0] - 1 / T)))

    # respiration rate m(T)
    temp_m = B[:, 1] * np.exp((-E[:, 1] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 1] / (Ed - E[:, 1])) * np.exp(Ed / k * (1 / Tp[:, 1] - 1 / T)))

    tt = np.column_stack((temp_u, temp_m))

    return tt



In [3]:
# MiCRM functions for later parameter generation 

def F_m(N, M, kw):
    
    if 'tt' in kw:      
        return kw['tt'][:, 1] 
    else:
        return np.full(N, 0.2)


def F_rho(N, M, kw):
    return np.ones(M)


def F_omega(N, M, kw):
    return np.ones(M)

# modular uptake and leakage 

def modular_uptake(N, M, N_modules, s_ratio, rng):
    assert N_modules <= M and N_modules <= N, "N_modules must be less than or equal to both M and N"

    # Baseline calculations
    sR = M // N_modules
    dR = M - (N_modules * sR)

    sC = N // N_modules
    dC = N - (N_modules * sC)

    # Get module sizes for M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[rng.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    # Get module sizes for N
    diffC = np.full(N_modules, sC, dtype=int)
    diffC[rng.random.choice(N_modules, dC, replace=False)] += 1
    mC = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffC) - diffC + 1), np.cumsum(diffC))]

    # Preallocate u matrix
    u = rng.random(N, M)

    # Apply scaling
    for x, y in zip(mC, mR):
        u[np.ix_(x, y)] *= s_ratio

    # Normalize each row
    for i in range(N):
        u[i, :] /= np.sum(u[i, :])

    return u


def F_u(N, M, kw): # incorporates modularity 

    u_pref = modular_uptake(
        N,
        M,
        kw["N_modules"],
        kw["s_ratio"],
        kw["rng"]
    )

    if "tt" in kw:
        u_sum = kw["tt"][:, 0]
    else:
        u_sum = np.full(N, 2.5)

    u = u_pref * u_sum[:, None]

    return u_pref, u


def modular_leakage(M, N_modules, s_ratio, λ, rng):
    assert N_modules <= M, "N_modules must be less than or equal to M"

    # Baseline
    sR = M // N_modules
    dR = M - (N_modules * sR)

    # Get module sizes and add to make to M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[rng.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    l = rng.random(M, M)

    for i, x in enumerate(mR):
        for j, y in enumerate(mR):
            if i == j or i + 1 == j:
                l[np.ix_(x, y)] *= s_ratio

    for i in range(M):
        l[i, :] = λ * l[i, :] / np.sum(l[i, :])

    return l


def F_l(N, M, kw): # incorporates modularity 
    rng = kw.get("rng", np.random)

    L = kw["L"]
    l = np.zeros((N, M, M))

    for i in range(N):
        l[i] = modular_leakage(
            M,
            kw["N_modules"],
            kw["s_ratio"],
            L[i],
            rng
        )

    return l

In [ ]:
# generate parameters for a single microbial community

def generate_params(N,
                     M,
                     f_m=F_m,
                     f_rho=F_rho,
                     f_omega=F_omega,
                     f_u=F_u,
                     f_l=F_l,
                     **kwargs):


    kw = dict(kwargs)
    B, E, Tp, D_resource = randtemp_param(N, kw) # Generate species-specific thermal traits

    tt = temp_trait(
            B,
            E,
            Tp,
            kw["T"],
            kw["Tr"],
            kw["Ed"]
        ) # tt = thermal traits; evaluate traits at the current temperature

    kw["tt"] = tt

 
    m = f_m(N, M, kw) 
    u_pref, u = f_u(N, M, kw) 
    l = f_l(N, M, kw)     


    lambda_ = np.sum(l, axis=2) 

 
    rho = f_rho(N, M, kw)
    omega = f_omega(N, M, kw)


    params = {
        'N': N,
        'M': M,
        'u': u,
        'u_pref':u_pref,
        'm': m,
        'l': l,
        'rho': rho,
        'omega': omega,
        'lambda': lambda_,
        'L': kw['L'],
        'B': B,
        'E': E,
        'Tp': Tp,
        'tt': tt,
        "D_resource": D_resource,
        "D_consumer": 0.01,
            }
   
    params.update(kwargs)

    return params 


In [ ]:
# incorporating spatial differences 
# function to generate many communities that can migrate and/or diffuse between each other


# User-defined settings for each patch (modify parameters as needed) 
patch_settings = [

    {
        "name": "cold_patch",

        # Community size
        "N": 50,
        "M": 25,

        # Thermal optimum
        "Tp_mean": 273.15 + 15,
        "Tp_sd": 5,

        # Baseline metabolism
        "B0_m": -1.60,
        "B0_CUE": 0.20,

        # Activation energies
        "E_mean": np.array([0.81, 0.57]),
        "E_var_scale": 0.1364,

        # Baseline-rate variation
        "B0_var_scale": 0.17,

        # Uptake-mortality correlation
        "rho_t": 0.5,

        # Leakage
        "L": np.full(50, 0.3),

        # modularity
        "N_modules": 2,
        "s_ratio": 10

    },

    {
        "name": "intermediate_patch",

        "N": 50,
        "M": 25,

        "Tp_mean": 273.15 + 20,
        "Tp_sd": 5,

        "B0_m": -1.45,
        "B0_CUE": 0.20,

        "E_mean": np.array([0.81, 0.57]),
        "E_var_scale": 0.1364,

        "B0_var_scale": 0.17,

        "rho_t": 0.5,

        "L": np.full(50, 0.3),

        "N_modules": 2,
        "s_ratio": 10
    },

    {
        "name": "warm_patch",

        "N": 50,
        "M": 25,

        "Tp_mean": 273.15 + 25,
        "Tp_sd": 5,

        "B0_m": -1.30,
        "B0_CUE": 0.20,

        "E_mean": np.array([0.81, 0.57]),
        "E_var_scale": 0.1364,

        "B0_var_scale": 0.17,

        "rho_t": 0.5,

        "L": np.full(50, 0.3),

        "N_modules": 2,
        "s_ratio": 10
    }

]



In [ ]:
# generate connectivity matrix between different patches to model their interactions

def create_connectivity_matrix(
    patch_positions,
    lambda_spatial=1.0,
    beta=0.0,
):
    """
    Create a distance-decay connectivity matrix

    Parameters involved:
    
    patch_positions : array-like coordinates of each patch
        E.g.,
            [0,1,2]
            or
            [[0,0],[1,0],[0,1]]

    lambda_spatial: float
        Controls how quickly connectivity decreases with distance

    beta: float
        Optional patch-degree weighting

    Returns
 
    A : ndarray (K x K)
        Connectivity matrix
    
    """

    patch_positions = np.asarray(patch_positions)

    K = len(patch_positions)

    if patch_positions.ndim == 1:
        patch_positions = patch_positions[:, None]

    # Euclidean distance matrix
    D = np.linalg.norm(
        patch_positions[:, None, :] -
        patch_positions[None, :, :],
        axis=2,
    )

    degree = np.ones(K) * (K - 1)

    A = np.zeros((K, K))

    for k in range(K):
        for j in range(K):

            if k == j:
                continue

            A[k, j] = (
                degree[j]**beta
                * np.exp(-lambda_spatial * D[k, j])
            )

    return A


# set up patch positions, entering coordinates for a defined number of patches 

patch_positions = np.array([
    [0,0],
    [1,0],
    [1,1],
]) # this example has 3 patches 

# create connectivity matrix for landscape

connectivity_matrix = create_connectivity_matrix(
    patch_positions,
    lambda_spatial=1.0,
)

In [ ]:
# finally, generate a landscape object that contains all the features of the landscape 
# including parameters for each individual MiCRM patch, their coordinate positions, and connectivity with each other

def generate_patch_landscape(
    patch_settings,
    patch_positions,
    connectivity_matrix,
    **kwargs,
    ):
    """
    Generate a spatial landscape consisting of multiple microbial communities.
    Each patch can have its own community size and environmental parameters.
    The returned landscape also stores the spatial positions and connectivity between patches.
    """

    patches = []

    for settings in patch_settings:

        settings = settings.copy()

        # Patch name
        patch_name = settings.pop("name")

        # Community size
        N = settings.pop("N")
        M = settings.pop("M")

        # Generate community parameters
        patch = generate_params(
            N=N,
            M=M,
            **kwargs,
            **settings,
        )

        patch["name"] = patch_name

        patches.append(patch)

    # Store everything describing the spatial landscape

    landscape = {

        "patches": patches,
        "positions": patch_positions,
        "connectivity": connectivity_matrix,
        "n_patches": len(patches),

    }

    return landscape


In [ ]:
# MiCRM solver function 
def dCdt_Rdt(t, y, structural):

    N = structural["N"]
    M = structural["M"]

    u = structural["u"]
    m = structural["m"]
    l = structural["l"]
    rho = structural["rho"]
    omega = structural["omega"]
    lambda_alpha = structural["lambda"]

    C = y[:N]
    R = y[N:]

    dCdt = np.zeros(N)
    dRdt = np.zeros(M)

    # Consumer dynamics
    for i in range(N):

        growth = sum(
            C[i] * R[alpha] * u[i, alpha] * (1 - lambda_alpha[i, alpha])
            for alpha in range(M)
        )

        dCdt[i] = growth - C[i] * m[i]

    # Resource dynamics
    for alpha in range(M):

        dRdt[alpha] = rho[alpha] - omega[alpha] * R[alpha]

        # Resource consumption
        dRdt[alpha] -= sum(
            C[i] * R[alpha] * u[i, alpha]
            for i in range(N)
        )

        # Leakage
        dRdt[alpha] += sum(
            sum(
                C[i] * R[beta] * u[i, beta] * l[i, beta, alpha]
                for beta in range(M)
            )
            for i in range(N)
        )

    return np.concatenate((dCdt, dRdt))



In [ ]:
# incorporating spatial-related transport to MiCRM 

def dCdt_Rdt_spatial(
    t,
    y,
    patches,
    connectivity,
):
    """
    Spatial MiCRM.

    patches = list of parameter dictionaries.

    connectivity = connectivity matrix (K x K)
    """

    K = len(patches)

    N = patches[0]["N"]
    M = patches[0]["M"]

    block = N + M

    dydt = np.zeros_like(y)

    for k in range(K):

        start = k * block
        end = start + block

        y_patch = y[start:end]

        # Local MiCRM dynamics
        local = dCdt_Rdt(
            t,
            y_patch,
            patches[k],
        )

        dydt[start:end] += local

    
    # Consumer migration
    

    for k in range(K):

        start_k = k * block

        Ck = y[start_k:start_k + N]

        D_consumer = patches[k]["D_consumer"]

        for j in range(K):

            if j == k:
                continue

            start_j = j * block

            Cj = y[start_j:start_j + N]

            dydt[start_k:start_k + N] += (
                D_consumer
                * connectivity[k, j]
                * (Cj - Ck)
            )

    # Resource transport


    for k in range(K):

        start_k = k * block

        Rk = y[start_k + N:start_k + N + M]

        D_resource = patches[k]["D_resource"]

        for j in range(K):

            if j == k:
                continue

            start_j = j * block

            Rj = y[start_j + N:start_j + N + M]

            dydt[start_k + N:start_k + N + M] += (
                D_resource
                * connectivity[k, j]
                * (Rj - Rk)
            )

    return dydt



Simulation code for temperature + spatial modelling: 

In [ ]:
def update_temperature(structural, T):
    """
    Evaluate an existing microbial community at a new temperature.
    Structural traits (B, E, Tp, uptake preferences, leakage, etc.)
    remain unchanged. Only the temperature-dependent parameters
    (uptake and mortality) are updated.
    """

    tt = temp_trait(
        structural["B"],
        structural["E"],
        structural["Tp"],
        T,
        structural["Tr"],
        structural["Ed"],
    )

    params = structural.copy()

    params["T"] = T
    params["tt"] = tt

    # Update temperature-dependent parameters
    params["m"] = tt[:, 1]
    params["u"] = structural["u_pref"] * tt[:, 0][:, None]

    return params


# output directory
outdir = "output"
os.makedirs(outdir, exist_ok=True)


# model parameters

Tr = 273.15 + 10
D_ref = 0.05 # reference diffusion rate
Ed = 3.5
temp_vals = np.linspace(273.15 + 10,
                        273.15 + 40,
                        16)
# if only modelling 1 temperature, modify temp_vals, e.g., temp_vals = [273.15 + 30]
tint = 1000
ttscle = 200
t_eval = np.linspace(0, tint, ttscle)

# run a single replicate 

def run_single_replicate(rep_id=1):

    rng = np.random.default_rng(111 + rep_id)

    # Generate spatial landscape

    landscape = generate_patch_landscape(
                patch_settings=patch_settings,
                patch_positions=patch_positions,
                connectivity_matrix=connectivity_matrix,
                Tr=Tr,
                Ed=Ed,
                rng=rng,
                )


    temperature_results = []

    # Evaluate the same landscape (including many mini-communities) across temperatures

    for T in temp_vals:

        patches = []

        for patch in landscape["patches"]:
            patches.append(update_temperature(patch, T))
        
        # set initial conditions 

        Y0 = []

        for patch in patches:

            C0 = np.full(patch["N"], 0.1)
            R0 = np.full(patch["M"], 1.0)

            Y0.extend(C0)
            Y0.extend(R0)

        Y0 = np.array(Y0)

        # solve MiCRM including spatial 
        
        sol = solve_ivp(
            dCdt_Rdt_spatial,
            (0, tint),
            Y0,
            args=(
                patches,
                landscape["connectivity"],
            ),
            t_eval=t_eval,
            method="LSODA",
            rtol=1e-4,
            atol=1e-7,
        )

        temperature_results.append({

            "temperature_K": T,
            "temperature_C": T - 273.15,
            "patches": patches,
            "solution": sol,

        })

    return {

        "replicate": rep_id,
        "landscape": landscape,
        "temperature_results": temperature_results,

    }


# to run a single microbial community: 

"""
single_run = run_single_replicate()

print("Simulation complete.")

sol_single = single_run["temperature_results"][0]["solution"]


# to repeat simulation across a number of microbial communities to obtain averages: 

multi_runs = []

for rep_id in range(1, 51): # 50 independently generated communities
    multi_runs.append(run_single_replicate(rep_id))

print("Simulation complete.")
"""


'\nmulti_runs = []\n\nfor rep_id in range(1, 51): # 50 independently generated communities\n    multi_runs.append(run_single_replicate(rep_id))\n\nprint("Simulation complete.")\n'